# 10 · Health indicators from the point-level data

**Why this notebook exists.** Notebook 02 shows that the handover's cycle-level features barely track SOH within a battery (median |ρ| ≈ 0.1, versus ≈ 1 for elapsed time), and notebook 04 shows that a persistence baseline beats the learned models. This notebook asks whether **physically meaningful health indicators (HIs)** extracted from the raw measurements do better.

**Source.** Every monthly cycling block in `cleaned_battery_data.csv` begins with the same standardised sequence in its first cycle: rest, constant-current (CC) charge at 1.46 A, constant-voltage (CV) charge, rest, 1C discharge, rest, then the grid-storage duty cycle. One row of indicators is extracted per block and aligned to the RPTs that bracket it (`src/health_indicators.py`, decision D14).

**Checks, in order:**
1. **Protocol leakage** — is any quantity set from a previous RPT result? (The 1C discharge current is.)
2. **Within-battery association** of each legitimate HI with SOH.
3. **Across batteries** — leave-one-battery-out prediction of the SOH at the RPT closing each block, compared with elapsed time alone and with the previous RPT's SOH.

## 1. Setup and extraction

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

from src import health_indicators as HI

dp.check_data_files(DATA_ROOT, required=["cleaned_battery_data.csv", "final_battery_health_dataset.csv"])
points = dp.read_dataset(DATA_ROOT / "cleaned_battery_data.csv")
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
blocks = HI.attach_rpt_labels(HI.extract_block_indicators(points), health)
groups = [g for g in dp.BATTERY_GROUPS if g in set(blocks[dp.GROUP_COL])]
print("blocks:", len(blocks), "| non-standard charge current:", int(blocks["nonstandard_charge"].sum()),
      "| blocks with missing steps:", int((blocks["missing_steps"].map(len) > 0).sum()),
      "| without a previous RPT:", int(blocks["soh_prev"].isna().sum()))
display(blocks[blocks["nonstandard_charge"] | (blocks["missing_steps"].map(len) > 0)][[dp.GROUP_COL, "Cycling_Test", "cc_current", "missing_steps"]])

## 2. Protocol leakage check

If the 1C discharge current was set from the capacity measured at the previous RPT, then the current, the discharged Ah and the discharge energy all encode the label. They are listed in `HI.PROTOCOL_SCALED` and excluded from every model below.

In [ ]:
scaling = HI.protocol_scaling(blocks)
display(scaling.round(4))
std = blocks[~blocks["nonstandard_charge"] & blocks["capacity_prev"].notna()]
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for g in groups:
    s = std[std[dp.GROUP_COL] == g]
    ax.plot(s["capacity_prev"], -s["dis_current"], "o", markersize=5, color=GROUP_COLORS[g], label=g)
lims = [std["capacity_prev"].min() - 0.02, std["capacity_prev"].max() + 0.02]
ax.plot(lims, lims, color="#8a8981", linewidth=1, linestyle="--", label="1C of previous RPT capacity")
ax.set_xlabel("Capacity at previous RPT (Ah)")
ax.set_ylabel("Monthly 1C discharge current (A)")
ax.set_title("Discharge current is set from the last RPT", loc="left")
ax.legend(frameon=False, fontsize=8, ncol=2)
plt.tight_layout()
savefig(fig, "10_protocol_scaling")
plt.show()

## 3. Within-battery association with SOH

Spearman ρ between each legitimate indicator and the SOH at the RPT closing the block, per battery. Elapsed time is shown as the reference: SOH falls almost monotonically with time, so any drifting quantity will correlate to some degree; a useful HI must do so *consistently across batteries* and add information beyond time (section 4).

In [ ]:
dq_cols = [c for c in blocks.columns if c.startswith("dq_")]
candidates = ["days_since_first_rpt", *HI.INDICATORS, *dq_cols]
use = blocks[~blocks["nonstandard_charge"] & blocks["soh_next"].notna()]
corr = pd.DataFrame({g: s[candidates + ["soh_next"]].corr(method="spearman")["soh_next"].drop("soh_next")
                     for g, s in use.groupby(dp.GROUP_COL)}).reindex(columns=groups)
order = ["days_since_first_rpt", *corr.drop(index="days_since_first_rpt").abs().median(axis=1).sort_values(ascending=False).index]
corr = corr.loc[order]
display(corr.assign(median=corr.median(axis=1)).round(2))

fig, ax = plt.subplots(figsize=(6.5, 7))
im = ax.imshow(corr.to_numpy(dtype=float), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(groups)), groups)
ax.set_yticks(range(len(corr)), ["elapsed days (reference)" if i == "days_since_first_rpt" else i for i in corr.index], fontsize=8)
for (i, j), v in np.ndenumerate(corr.to_numpy(dtype=float)):
    if np.isfinite(v):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.6 else "#1a1a19")
ax.axhline(0.5, color="#1a1a19", linewidth=1)
ax.set_title("Health indicators: within-battery Spearman ρ with SOH", loc="left")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.6, label="Spearman ρ")
plt.tight_layout()
savefig(fig, "10_indicator_correlation")
plt.show()

In [ ]:
best = corr.drop(index="days_since_first_rpt").median(axis=1).abs().idxmax()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for g in groups:
    s = use[use[dp.GROUP_COL] == g].sort_values("block_start")
    axes[0].plot(s["days_since_first_rpt"], s[best], marker="o", markersize=4, color=GROUP_COLORS[g], label=g)
    axes[1].plot(s[best], s["soh_next"], "o", markersize=5, color=GROUP_COLORS[g], label=g)
axes[0].set_xlabel("days since first RPT"); axes[0].set_ylabel(f"{best} (Ah)")
axes[0].set_title(f"Strongest indicator over time: {best}", loc="left")
axes[1].set_xlabel(f"{best} (Ah)"); axes[1].set_ylabel("SOH at closing RPT (%)")
axes[1].set_title("Indicator vs. SOH", loc="left")
axes[1].legend(frameon=False, fontsize=8, ncol=3)
plt.tight_layout()
savefig(fig, "10_best_indicator")
plt.show()

## 4. Do the indicators predict SOH on an unseen battery?

Leave-one-battery-out ridge regression (preprocessing and penalty fitted on the other five batteries) of the SOH at the RPT closing each block. References: elapsed time alone, and persistence (the previous RPT's SOH, which is known when the block starts). Blocks with a non-standard protocol or no previous RPT are excluded.

In [ ]:
d = blocks[~blocks["nonstandard_charge"] & blocks["soh_prev"].notna()].copy()
feature_sets = {
    "elapsed time": ["days_since_first_rpt"],
    "all indicators": list(HI.INDICATORS) + dq_cols,
    f"{best}": [best],
    f"{best} + time": [best, "days_since_first_rpt"],
    "all indicators + time": list(HI.INDICATORS) + dq_cols + ["days_since_first_rpt"],
}
lobo = HI.leave_one_battery_out(d, feature_sets)
lobo["persistence (previous RPT)"] = d.groupby(dp.GROUP_COL).apply(
    lambda s: float(np.sqrt(np.mean((s["soh_prev"] - s["soh_next"]) ** 2))), include_groups=False)
lobo.loc["mean", "persistence (previous RPT)"] = lobo.drop(index="mean")["persistence (previous RPT)"].mean()
display(lobo.round(3))

m = lobo.loc["mean"].sort_values()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(m.index, m.values, color=["#2a78d6" if "indicator" in k or k.startswith("dq") else "#8a8981" for k in m.index], height=0.6)
for i, v in enumerate(m.values):
    ax.text(v, i, f" {v:.2f}", va="center", fontsize=9, color="#3d3d3a")
ax.invert_yaxis()
ax.set_xlabel("Mean held-out-battery RMSE (SOH pp)")
ax.set_title("Leave-one-battery-out SOH estimation", loc="left")
ax.grid(axis="y", visible=False)
plt.tight_layout()
savefig(fig, "10_lobo_rmse")
plt.show()

## 5. Save results

In [ ]:
import json
out = RESULTS_DIR / "health_indicators.json"
RESULTS_DIR.mkdir(exist_ok=True)
payload = {"environment": dp.environment_info(REPO_ROOT), "n_blocks": int(len(blocks)),
           "protocol_scaling": scaling.reset_index().to_dict(orient="records"),
           "within_battery_spearman": corr.reset_index().to_dict(orient="records"),
           "lobo_rmse": lobo.reset_index().to_dict(orient="records"), "best_indicator": best}
out.write_text(json.dumps(payload, indent=2, default=str))
print("wrote", out.relative_to(REPO_ROOT))

## 6. Interpretation

* **Leakage:** the monthly 1C discharge current equals ≈ 0.997 × the previous RPT capacity for every battery. The discharge current, discharged Ah and energy are therefore excluded; so should the handover's `Current_Min` and discharge-capacity columns in any model (they partly carry the same scaling).
* **Association:** partial-charge indicators (charge accumulated in a fixed voltage window during the CC charge) track SOH in the physically expected direction within batteries.
* **Across batteries:** if no indicator set beats elapsed time or persistence on held-out batteries, the limiting factor is the data, not the method: all six cells fade by only 3–5 pp along nearly identical trajectories, so time explains almost all variance and monthly RPTs make persistence very strong. Estimating the *current* SOH is then not a meaningful benchmark on this dataset; forecasting future SOH, or adding datasets with larger and more diverse fade, is.